In [0]:
# 0) Preparación: cargar silver y definir una agregación de ejemplo
from pyspark.sql import functions as F
import time

silver = spark.table("workspace.credito_riesgo_lab.clientes_silver")

# Definimos una "acción" típica para medir (con .count() forzamos la ejecución)
def agregacion(df):
    return (
        df.groupBy("historial_pagos")
          .agg(F.round(F.avg("ingresos"), 2).alias("ingreso_medio"))
          .count()
    )

# SIN cache: cada llamada recalcula TODO desde el disco (leer + filtrar + agregar)
t0 = time.time(); agregacion(silver); t1 = time.time()
t2 = time.time(); agregacion(silver)
print(f"1ª vez: {t1-t0:.2f}s | 2ª vez: {t2-t1:.2f}s  ← recalcula de nuevo desde cero")
print("Nota: con solo 10.000 filas la diferencia es pequeña; con millones de filas es enorme.")

In [0]:
# 1) cache() — guardar en memoria para NO recalcular lo mismo
# Regla: cachear SOLO si el DataFrame se usa varias veces; liberar con unpersist() al acabar.
# LECCIÓN (Spark Connect): tras unpersist() ese DataFrame ya NO se puede reutilizar.
# Por eso la demo usa una copia desechable (leída de nuevo de la tabla), para no romper
# las celdas siguientes que siguen usando 'silver'.
silver_demo = spark.table("workspace.credito_riesgo_lab.clientes_silver")

def agregacion_demo(df):
    return (
        df.groupBy("historial_pagos")
          .agg(F.round(F.avg("ingresos"), 2).alias("ingreso_medio"))
          .count()
    )

silver_cacheado = silver_demo.cache()

t0 = time.time(); agregacion_demo(silver_cacheado); t1 = time.time()   # 1ª vez: calcula Y deja en memoria
t2 = time.time(); agregacion_demo(silver_cacheado); t3 = time.time()   # 2ª vez: lee de memoria
print(f"Con cache | 1ª vez: {t1-t0:.2f}s | 2ª vez: {t2-t1:.2f}s (debería ser más rápida)")

silver_cacheado.unpersist()   # liberar memoria cuando ya no se necesita

In [0]:
# 2) Particiones: cómo Spark reparte los datos entre las máquinas del cluster
# repartition(n): redistribuye SIEMPRE (shuffle por la red, caro; sirve para subir o repartir por clave)
# coalesce(n): solo fusiona particiones existentes (barato; ideal para reducir antes de escribir)

# ¿Cuántos ficheros tiene la tabla en disco? (DESCRIBE DETAIL nos lo dice)
detalle_antes = spark.sql("DESCRIBE DETAIL workspace.credito_riesgo_lab.clientes_silver") \
    .select("numFiles", "sizeInBytes")
display(detalle_antes)

# coalesce(4): fusionar en 4 trozos antes de escribir → menos ficheros pequeños en disco
(
    silver.coalesce(4).write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.credito_riesgo_lab.clientes_silver_opt")
)

detalle_despues = spark.sql("DESCRIBE DETAIL workspace.credito_riesgo_lab.clientes_silver_opt") \
    .select("numFiles", "sizeInBytes")
display(detalle_despues)

In [0]:
# 3) broadcast() — join eficiente cuando una tabla es PEQUEÑA
# Sin broadcast (SortMergeJoin): Spark mueve AMBAS tablas por la red (shuffle), muy caro.
# Con broadcast (BroadcastHashJoin): la tabla pequeña se copia a cada ejecutor y el join es local.
dim_sector = spark.sql("""
    SELECT sector,
           CASE sector WHEN 'tecnologia' THEN 'bajo'
                       WHEN 'servicios'  THEN 'medio'
                       ELSE 'alto' END AS riesgo_sectorial
    FROM (SELECT DISTINCT sector FROM workspace.credito_riesgo_lab.clientes_silver)
""")

join_broadcast = silver.join(F.broadcast(dim_sector), "sector")

# El plan físico revela el tipo de join: busca "BroadcastHashJoin" en la salida
join_broadcast.explain()

In [0]:
# 4) partitionBy al escribir — organizar la tabla en carpetas por sector
# Ventaja (partition pruning): si luego filtras por sector, Spark solo lee la carpeta necesaria
(
    silver.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("sector")
    .saveAsTable("workspace.credito_riesgo_lab.clientes_silver_por_sector")
)

# Al filtrar por sector, el plan lee solo esa partición (busca "PartitionFilters" en el plan)
consulta = spark.sql("""
    SELECT COUNT(*) AS clientes_retail
    FROM workspace.credito_riesgo_lab.clientes_silver_por_sector
    WHERE sector = 'retail'
""")
consulta.show()
consulta.explain()

In [0]:
# 5) Chuleta de optimización (nivel estudiante)
print("""
REGLAS DE ORO DE OPTIMIZACIÓN:
1. cache()/persist() solo si el DataFrame se usa VARIAS veces; unpersist() al acabar.
2. coalesce() para reducir particiones (barato); repartition() solo si necesitas MÁS
   particiones o repartir por una clave.
3. broadcast() cuando una tabla del join es pequeña (cabe en memoria de cada ejecutor).
4. partitionBy al escribir tablas grandes que suelen filtrarse por esa columna.
5. Menos shuffle = menos red = menos tiempo = menos coste de máquinas.
""")

# 03 · Optimización de recursos con PySpark

**Objetivo:** aprender técnicas básicas para que tus jobs usen **menos memoria, menos CPU y menos tiempo** (es decir, menos coste de máquinas):

1. `cache()` / `persist()` — no recalcular lo mismo varias veces
2. `repartition()` vs `coalesce()` — cuántas particiones (trozos) tiene el DataFrame
3. `broadcast()` — joins sin mover datos gigantes por la red
4. `partitionBy` al escribir — organizar la tabla en disco por carpeta
5. Medir: contar particiones y comparar tiempos

> Ejecuta antes `00_datos_sinteticos` y `01_medallion_pipeline`.